In [105]:
import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

In [1]:
API_KEY_PATH ='/home/project_yujin/high_project/sprintda03-yujin.json'

def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name

In [107]:
accounts_attendance = get_df('votes','accounts_attendance')
accounts_attendance = accounts_attendance[['id', 'user_id', 'attendance_date_list']]

## 분석에 적합한 형태로 프레임 변환

In [108]:
accounts_attendance['attendance_date_list'] = accounts_attendance['attendance_date_list'].apply(ast.literal_eval) # literal_eval 으로변환
expended_attendance = accounts_attendance.explode('attendance_date_list').reset_index(drop=True) # 데이터 프래암 형식 변환

In [109]:
expended_attendance.rename(columns={'attendance_date_list':'attendance_date'}, inplace=True)

## 변환후 일치한지 확인

In [110]:
# 각 attendance_date_list의 길이를 구한 뒤 총합 계산
attendance_lengths = accounts_attendance['attendance_date_list'].apply(len)
attendance_lengths.sum()

2222327

In [111]:
expended_attendance['attendance_date'].notna().sum()

2222327

## info

In [113]:
expended_attendance['attendance_date'] = pd.to_datetime(expended_attendance['attendance_date'])

In [114]:
expended_attendance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2243272 entries, 0 to 2243271
Data columns (total 3 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               int64         
 1   user_id          int64         
 2   attendance_date  datetime64[ns]
dtypes: datetime64[ns](1), int64(2)
memory usage: 51.3 MB


## nunique

In [78]:
def get_nunique(df):
    return pd.merge(df.nunique().reset_index(name='nunique'),
                    ((df.nunique()/len(df))*100).reset_index(name='nunique_ratio'),
                    on='index')

In [79]:
get_nunique(expended_attendance)

,index,nunique,nunique_ratio
0,id,349637,15.586028
1,user_id,349637,15.586028
2,attendance_date,333,0.014844


## 중복값

In [45]:
expended_attendance[['user_id', 'attendance_date']].duplicated().sum()

0

## 결측값

In [80]:
def get_isna(df):
    return pd.merge(df.isna().sum().reset_index(name='isna'), 
                    (df.isna().mean()*100).reset_index(name='isna_ratio'), 
                    on='index')

In [81]:
get_isna(expended_attendance)

,index,isna,isna_ratio
0,id,0,0.000000
1,user_id,0,0.000000
2,attendance_date,20945,0.933681


## 1번 테이블 내에서 attendance_date 결측값 분석

In [ ]:
date_isna = expended_attendance[expended_attendance['attendance_date'].isna()]
user_date_isna = date_isna['user_id'].unique().tolist()

In [93]:
# 출석날자가 결측값인 유저데이터 추출
accounts_attendance[accounts_attendance['user_id'].isin(user_date_isna)]

,id,user_id,attendance_date_list
19,20,1484400,[]
237,243,1284925,[]
306,312,1451267,[]
333,339,1483247,[]
366,375,1173480,[]
...,...,...,...
349618,360487,1162275,[]
349622,360491,935331,[]
349628,360497,1127910,[]
349629,360498,851491,[]


In [100]:
expended_user_date_isna = accounts_attendance[accounts_attendance['user_id'].isin(user_date_isna)]
len(expended_user_date_isna)

20945

In [98]:
expended_user_date_isna['user_id'].nunique()

20945

## 출석수 분포


In [116]:
expended_attendance['attendance_date'].min(), expended_attendance['attendance_date'].max()

(Timestamp('2023-05-27 00:00:00'), Timestamp('2024-05-09 00:00:00'))

In [121]:
expended_attendance['peroid_M'] = expended_attendance['attendance_date'].dt.to_period('M')
expended_attendance

,id,user_id,attendance_date,peroid_M
0,1,1446852,2023-05-27,2023-05
1,1,1446852,2023-05-28,2023-05
2,1,1446852,2023-05-29,2023-05
3,1,1446852,2023-05-30,2023-05
4,1,1446852,2023-06-03,2023-06
...,...,...,...,...
2243267,360501,897005,NaT,NaT
2243268,360502,1407059,2024-05-09,2024-05
2243269,360503,1583727,2024-05-09,2024-05
2243270,360504,1392372,2024-05-09,2024-05


In [123]:
expended_attendance.groupby(['user_id', 'peroid_M'])['attendance_date'].mean()

user_id  peroid_M
832151   2023-05    2023-05-29 00:00:00
832340   2023-06    2023-06-05 12:00:00
832986   2023-05    2023-05-28 00:00:00
833041   2023-05    2023-05-29 00:00:00
         2023-06    2023-06-14 16:00:00
                            ...        
1583708  2024-05    2024-05-06 00:00:00
1583710  2024-05    2024-05-05 00:00:00
1583715  2024-05    2024-05-05 00:00:00
1583727  2024-05    2024-05-09 00:00:00
1583730  2024-05    2024-05-09 00:00:00
Name: attendance_date, Length: 626717, dtype: datetime64[ns]

In [133]:
expended_attendance.groupby(['user_id'])['attendance_date'].nunique()

user_id
832151      1
832340      2
832986      1
833041     53
833112      3
           ..
1583710     1
1583711     0
1583715     1
1583727     1
1583730     1
Name: attendance_date, Length: 349637, dtype: int64

In [ ]:
expended_attendance['user_id'].value_counts()

In [ ]:
expended_attendance.groupby(['user_id'])['attendance_date'].agg(date_min='min', date_max='max', date_count='nunique')

,date_min,date_max,date_count
user_id,,,
832151,2023-05-29,2023-05-29,1
832340,2023-06-05,2023-06-06,2
832986,2023-05-28,2023-05-28,1
833041,2023-05-27,2024-02-09,53
833112,2023-09-18,2023-09-24,3
...,...,...,...
1583710,2024-05-05,2024-05-05,1
1583711,NaT,NaT,0
1583715,2024-05-05,2024-05-05,1


In [ ]:
user_attendance_stats = (
    expended_attendance.groupby(['user_id', 'peroid_M'])['attendance_date'].nunique()
    .groupby('user_id').agg(
        total_attendance='sum',
        mean_attendance='mean',
        min_attendance='min',
        max_attendance='max'
        )
    .reset_index()
)

In [127]:
user_attendance_stats

,user_id,total_attendance,mean_attendance,min_attendance,max_attendance
0,832151,1,1.0,1,1
1,832340,2,2.0,2,2
2,832986,1,1.0,1,1
3,833041,53,5.9,1,18
4,833112,3,3.0,3,3
...,...,...,...,...,...
328687,1583708,1,1.0,1,1
328688,1583710,1,1.0,1,1
328689,1583715,1,1.0,1,1
328690,1583727,1,1.0,1,1
